# Delta Lake Optimizations
Compacts small files and Z-orders FactSales for faster queries, demonstrates
time travel, and vacuums old file versions to control storage cost.
Runs after every successful pipeline run.

In [0]:
dbutils.widgets.text("catalog_name", "workspace", "Catalog Name")
dbutils.widgets.text("schema_prefix", "retail", "Schema Prefix")

In [0]:
catalog = dbutils.widgets.get("catalog_name")
schema_prefix = dbutils.widgets.get("schema_prefix")

gold_db = f"{catalog}.{schema_prefix}_gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"USE {gold_db}")

print("Optimizing tables in:", gold_db)

In [0]:
print("Optimizing FactSales...")
spark.sql(f"OPTIMIZE {gold_db}.FactSales ZORDER BY (order_purchase_date_key, product_key)")
print("Done.")

In [0]:
display(spark.sql(f"DESCRIBE HISTORY {gold_db}.FactSales").select("version", "timestamp", "operation"))

In [0]:
from datetime import datetime, timedelta

safe_days_back = 6  # stays under the 7-day (168 hour) VACUUM retention window
target_ts = (datetime.utcnow() - timedelta(days=safe_days_back)).strftime("%Y-%m-%d %H:%M:%S")

try:
    df_old = spark.read.format("delta").option("timestampAsOf", target_ts).table(f"{gold_db}.FactSales")
    print(f"Time traveled to ~{safe_days_back} days ago ({target_ts})")
    print(f"Row count then: {df_old.count()}")
    print(f"Current row count: {spark.table(f'{gold_db}.FactSales').count()}")
except Exception as e:
    print(f"No version available that far back yet — table is younger than {safe_days_back} days. Skipping time travel demo for now.")

In [0]:
result = spark.sql(f"VACUUM {gold_db}.FactSales")
display(result)
print("Vacuum complete (default 7-day retention kept).")